In [1]:
from app.agent.graph import graph

result = graph.invoke({
    "question": "پرفروش ترین محصول چیست؟"
})

print(result)

{'question': 'پرفروش ترین محصول چیست؟', 'mode': 'full', 'intro_message': 'کوئری برای یافتن پرفروش\u200cترین محصول آماده شده است.', 'sql_message': 'این کوئری نام محصولی را که بیشترین تعداد فروش را داشته است، به همراه مجموع تعداد فروش آن نمایش می\u200cدهد. نتیجه این کوئری در ادامه آمده است.', 'sql': 'SELECT p.product_name, SUM(oi.quantity) AS total_quantity_sold\nFROM sales.order_items oi\nJOIN production.products p ON oi.product_id = p.product_id\nGROUP BY p.product_name\nORDER BY total_quantity_sold DESC\nLIMIT 1;', 'result': [{'product_name': 'نوکیا 3310', 'total_quantity_sold': 30}], 'analysis': 'محصول پرفروش در این مجموعه داده، "نوکیا 3310" است.\n\nاین محصول با مجموع فروش 30 واحد، بیشترین تعداد فروش را در بین محصولات دیگر داشته است.\n\nنتیجه نشان می\u200cدهد که "نوکیا 3310" در صدر لیست محصولات پرفروش قرار دارد.', 'error': None}


In [1]:
from app.agent.graph import graph

inputs = {
    "question": "پرفروش ترین محصول چیست؟"
}

for event in graph.stream(inputs):
    print("="*50)
    print(event)

{'intent': {'mode': 'full'}}
{'full': {'intro_message': 'کوئری زیر برای یافتن پرفروش\u200cترین محصول آماده شده است.', 'sql': 'SELECT p.product_name, SUM(oi.quantity) AS total_quantity_sold\nFROM production.products p\nJOIN sales.order_items oi ON p.product_id = oi.product_id\nGROUP BY p.product_name\nORDER BY total_quantity_sold DESC\nLIMIT 1;', 'sql_message': 'این کوئری نام محصولی را که بیشترین تعداد فروش را داشته است، به همراه مجموع تعداد فروش آن نمایش می\u200cدهد. نتیجه این کوئری در ادامه آمده است.'}}
{'execute_sql': {'result': [{'product_name': 'Cannondale Topstone', 'total_quantity_sold': 2}], 'error': None}}
{'analyzer': {'analysis': 'محصول "Cannondale Topstone" به عنوان پرفروش\u200cترین محصول شناخته شده است.\n\nتعداد کل فروش این محصول برابر با ۲ واحد می\u200cباشد.\n\nاین نتیجه نشان می\u200cدهد که در میان محصولات موجود، "Cannondale Topstone" بیشترین تعداد فروش را داشته است.'}}


In [2]:
from app.agent.graph import graph

inputs = {
    "question": "سلام"
}

for event in graph.stream(inputs):
    print("="*50)
    print(event)

{'intent': {'mode': 'chat'}}
{'chat': {'message': 'سلام! من اینجا هستم تا به شما در مورد سوالات دیتابیس و SQL کمک کنم. چطور می\u200cتونم کمکتون کنم؟'}}


In [3]:
from app.agent.graph import graph

inputs = {
    "question": "فقط کوعری 3 مشتری برتر رو بده"
}

for event in graph.stream(inputs):
    print("="*50)
    print(event)

{'intent': {'mode': 'sql'}}
{'sql': {'sql': 'SELECT c.customer_id, c.first_name, c.last_name, SUM(oi.list_price * oi.quantity * (1 - oi.discount)) AS total_spent\nFROM sales.customers c\nJOIN sales.orders o ON c.customer_id = o.customer_id\nJOIN sales.order_items oi ON o.order_id = oi.order_id\nGROUP BY c.customer_id, c.first_name, c.last_name\nORDER BY total_spent DESC\nLIMIT 3;'}}


In [4]:
from app.agent.graph import graph

inputs = {
    "question": "فقط نتیجه سه مشتری برتر رو بده"
}

for event in graph.stream(inputs):
    print("="*50)
    print(event)

{'intent': {'mode': 'result'}}
{'sql': {'sql': 'SELECT c.customer_id, c.first_name, c.last_name, SUM(oi.list_price * oi.quantity * (1 - oi.discount)) AS total_spent\nFROM sales.customers c\nJOIN sales.orders o ON c.customer_id = o.customer_id\nJOIN sales.order_items oi ON o.order_id = oi.order_id\nGROUP BY c.customer_id, c.first_name, c.last_name\nORDER BY total_spent DESC\nLIMIT 3;'}}
{'execute_sql': {'result': [{'customer_id': 4, 'first_name': 'Maryam', 'last_name': 'Rad', 'total_spent': Decimal('5500.0000')}, {'customer_id': 2, 'first_name': 'Sara', 'last_name': 'Ahmadi', 'total_spent': Decimal('4200.0000')}, {'customer_id': 5, 'first_name': 'Omid', 'last_name': 'Sadeghi', 'total_spent': Decimal('3906.0000')}], 'error': None}}
{'analyzer': {'analysis': 'مشتری اول با بیشترین میزان خرید، مریم راد با شناسه مشتری ۴ است که مجموع خریدهای او به مبلغ ۵۵۰۰ واحد می\u200cرسد.\n\nمشتری دوم سارا احمدی با شناسه مشتری ۲ است که مجموع خریدهای او ۴۲۰۰ واحد است.\n\nمشتری سوم امید صادقی با شناسه مشتری 

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from app.core.config import settings
from app.agent.schemas  import (
    IntentOutput,
    ChatOutput
)


#LLM Setup 

llm = ChatOpenAI(
    base_url="https://api.gapgpt.app/v1",
    api_key=settings.OPENAI_API_KEY,
    model="gpt-4o",
    temperature=0,
    streaming=False
)

streaming_llm = ChatOpenAI(
    base_url="https://api.gapgpt.app/v1",
    api_key=settings.OPENAI_API_KEY,
    model="gpt-4o",
    temperature=0,
    streaming=True
)


#Prompts 

intent_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an intent classifier for a SQL agent.

Classify the user request into one of these modes:

chat   → greeting or unrelated to database
sql    → user explicitly asks for SQL query only
result → user wants only the raw result
full   → default for any data question

Important:
If the user asks about data, ranking, statistics, counts, etc.,
and does NOT explicitly request SQL only,
you MUST return full.
"""),
    ("human", "{question}")
])

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a SQL intelligent assistant connected to the user's database.
You MUST always respond in Persian (Farsi) language only. Never respond in English or any other language.

Mirror the user's tone and formality:
- If they're casual and warm, be friendly and conversational.
- If they're direct and brief, keep it short and to the point.
- If they greet you, greet back warmly, then briefly mention what you do.
- If they don't greet you, never greet them.
- If they jump straight to questions, skip the pleasantries and get to work.

Your role:
You help users query their SQL database through natural language. Users can ask questions and receive:
- SQL queries
- Query results in table format
- Analysis and insights

Stay focused:
Only answer SQL and database-related questions.
If asked something irrelevant, politely redirect in Persian: "من فقط می‌توانم در مورد سوالات دیتابیس و SQL کمک کنم."

"""),
    ("human", "{question}")
])


intent_chain = intent_prompt | llm.with_structured_output(IntentOutput)
chat_chain = chat_prompt | llm.with_structured_output(ChatOutput)


In [6]:
from typing import Dict, List, Any
from app.agent.schemas.states  import AgentState
from app.agent.chains import (
    intent_chain,
    chat_chain,
    sql_chain,
    full_chain,
    analyzer_chain,
)
from app.core.database.clientdb import get_db_schema_text , run_sql_query


#Nodes

def intent_node(state: AgentState):
    result = intent_chain.invoke({"question": state["question"]})
    return {"mode": result.mode}


def router(state: AgentState):
    return state["mode"]


def chat_node(state: AgentState):
    result = chat_chain.invoke({"question": state["question"]})
    return {"message": result.message}




In [7]:
from langgraph.graph import StateGraph, END

from app.agent.schemas.states  import AgentState
from app.agent.nodes import (
    intent_node,
    router,
    chat_node,
    sql_node,
    full_node,
    execute_sql_node,
    after_sql_router,
    analyzer_node,
)


#Graph

def build_graph():
    builder = StateGraph(AgentState)

    builder.add_node("intent", intent_node)
    builder.add_node("chat", chat_node)


    builder.set_entry_point("intent")

    builder.add_conditional_edges(
        "intent",
        router,
        {
            "chat": "chat"
        }
    )
    return builder.compile()


graph = build_graph()

In [9]:
inputs = {
    "question": "سلام"
}
graph.invoke(inputs)

{'question': 'سلام',
 'mode': 'chat',
 'message': 'سلام! من می\u200cتونم به شما در نوشتن کوئری\u200cهای SQL و تحلیل داده\u200cها کمک کنم. چطور می\u200cتونم کمکتون کنم؟'}

In [10]:
inputs = {
    "question": "سلام"
}

for event in graph.stream(inputs):
    print("="*50)
    print(event)

{'intent': {'mode': 'chat'}}
{'chat': {'message': 'سلام! من اینجا هستم تا به شما در نوشتن و اجرای کوئری\u200cهای SQL کمک کنم. چطور می\u200cتونم کمکتون کنم؟'}}


In [8]:
import asyncio
from typing import Optional, Any
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END
from langgraph.types import StreamWriter

# ─────────────────────────────────────────
# Config
# ─────────────────────────────────────────
BASE_URL = "https://api.gapgpt.app/v1"
API_KEY  = "sk-s8KnoW59PPxeHBvyzENeVoEiH2QbiNm1PxJt20H586up5p8n"
MODEL    = "gpt-4o"

# ─────────────────────────────────────────
# LLMs
# ─────────────────────────────────────────
llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
    streaming=False,
)

streaming_llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
    streaming=True,
)

# ─────────────────────────────────────────
# Schemas
# ─────────────────────────────────────────
class IntentOutput(BaseModel):
    mode: str = Field(description="chat | sql | result | full")

# ─────────────────────────────────────────
# State
# ─────────────────────────────────────────
class TestState(TypedDict):
    question: str
    mode:     Optional[str]
    message:  Optional[str]

# ─────────────────────────────────────────
# Prompts & Chains
# ─────────────────────────────────────────
intent_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an intent classifier for a SQL agent.
Classify the user request into one of these modes:
chat   → greeting or unrelated to database
sql    → user explicitly asks for SQL query only
result → user wants only the raw result
full   → default for any data question
"""),
    ("human", "{question}"),
])

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a SQL intelligent assistant.
You MUST always respond in Persian (Farsi) language only.
Only answer SQL and database-related questions.
"""),
    ("human", "{question}"),
])

intent_chain = intent_prompt | llm.with_structured_output(IntentOutput)
chat_chain   = chat_prompt   | streaming_llm   # بدون structured_output

# ─────────────────────────────────────────
# Nodes
# ─────────────────────────────────────────
def test_intent_node(state: TestState):
    result = intent_chain.invoke({"question": state["question"]})
    return {"mode": result.mode}

def test_router(state: TestState):
    return state["mode"]

def test_chat_node(state: TestState, writer: StreamWriter):
    full_message = ""

    for chunk in chat_chain.stream({"question": state["question"]}):
        token = chunk.content
        if token:
            full_message += token
            writer({"type": "token", "value": token})

    return {"message": full_message}

# ─────────────────────────────────────────
# Graph
# ─────────────────────────────────────────
def build_test_graph():
    builder = StateGraph(TestState)

    builder.add_node("test_intent", test_intent_node)
    builder.add_node("test_chat",   test_chat_node)

    builder.set_entry_point("test_intent")

    builder.add_conditional_edges(
        "test_intent",
        test_router,
        {"chat": "test_chat"},
    )

    builder.add_edge("test_chat", END)

    return builder.compile()

test_graph = build_test_graph()

# ─────────────────────────────────────────
# Run
# ─────────────────────────────────────────
inputs = {"question": "سلام"}

for event in test_graph.stream(inputs, stream_mode=["updates", "custom"]):
    mode, data = event

    if mode == "updates":
        print(data)

    elif mode == "custom":
        print({"chat": {"custom": data}})

{'test_intent': {'mode': 'chat'}}
{'chat': {'custom': {'type': 'token', 'value': 'سلام'}}}
{'chat': {'custom': {'type': 'token', 'value': '!'}}}
{'chat': {'custom': {'type': 'token', 'value': ' چ'}}}
{'chat': {'custom': {'type': 'token', 'value': 'طور'}}}
{'chat': {'custom': {'type': 'token', 'value': ' می'}}}
{'chat': {'custom': {'type': 'token', 'value': '\u200cتوان'}}}
{'chat': {'custom': {'type': 'token', 'value': 'م'}}}
{'chat': {'custom': {'type': 'token', 'value': ' در'}}}
{'chat': {'custom': {'type': 'token', 'value': ' زمینه'}}}
{'chat': {'custom': {'type': 'token', 'value': ' SQL'}}}
{'chat': {'custom': {'type': 'token', 'value': ' یا'}}}
{'chat': {'custom': {'type': 'token', 'value': ' پای'}}}
{'chat': {'custom': {'type': 'token', 'value': 'گاه'}}}
{'chat': {'custom': {'type': 'token', 'value': '\u200c'}}}
{'chat': {'custom': {'type': 'token', 'value': 'د'}}}
{'chat': {'custom': {'type': 'token', 'value': 'اده'}}}
{'chat': {'custom': {'type': 'token', 'value': ' به'}}}
{'cha

In [13]:
import asyncio
from typing import Optional, Any
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END
from langgraph.types import StreamWriter

# ─────────────────────────────────────────
# Config
# ─────────────────────────────────────────
BASE_URL = "https://api.gapgpt.app/v1"
API_KEY  = "sk-s8KnoW59PPxeHBvyzENeVoEiH2QbiNm1PxJt20H586up5p8n"
MODEL    = "gpt-4o"

# ─────────────────────────────────────────
# LLMs
# ─────────────────────────────────────────
llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
    streaming=False,
)

streaming_llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
    streaming=True,
)

# ─────────────────────────────────────────
# Schemas
# ─────────────────────────────────────────
class IntentOutput(BaseModel):
    mode: str = Field(description="chat | sql | result | full")

# ─────────────────────────────────────────
# State
# ─────────────────────────────────────────
class AgentState(TypedDict):
    question:      str
    mode:          Optional[str]
    message:       Optional[str]   # chat node final message
    sql:           Optional[str]   # generated SQL
    result:        Optional[Any]   # query execution result
    intro_message: Optional[str]   # full mode intro
    sql_message:   Optional[str]   # full mode sql explanation

# ─────────────────────────────────────────
# Helpers  (stubs – replace with your real implementations)
# ─────────────────────────────────────────
def get_db_schema_text() -> str:
    """Return the database schema as plain text."""
    # TODO: replace with your real schema loader
    return """
    TABLE users (id SERIAL PRIMARY KEY, name TEXT, email TEXT, created_at TIMESTAMP);
    TABLE orders (id SERIAL PRIMARY KEY, user_id INT REFERENCES users(id), total NUMERIC, created_at TIMESTAMP);
    TABLE products (id SERIAL PRIMARY KEY, name TEXT, price NUMERIC, stock INT);
    TABLE order_items (id SERIAL PRIMARY KEY, order_id INT REFERENCES orders(id), product_id INT REFERENCES products(id), quantity INT);
    """

def run_sql_query(sql: str) -> Any:
    """Execute the SQL and return the result."""
    # TODO: replace with your real DB executor
    return [{"id": 1, "name": "Alice", "total": 250.0}]

# ─────────────────────────────────────────
# Prompts & Chains
# ─────────────────────────────────────────

# --- Intent ---
intent_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an intent classifier for a SQL agent.
Classify the user request into one of these modes:
chat   → greeting or unrelated to database
sql    → user explicitly asks for SQL query only
result → user wants only the raw result (no explanation)
full   → default for any data question (intro + sql + explanation + analysis)
"""),
    ("human", "{question}"),
])
intent_chain = intent_prompt | llm.with_structured_output(IntentOutput)

# --- Chat ---
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a SQL intelligent assistant.
You MUST always respond in Persian (Farsi) language only.
Only answer SQL and database-related questions.
"""),
    ("human", "{question}"),
])
chat_chain = chat_prompt | streaming_llm

# --- SQL generation (used in sql, result, full modes) ---
sql_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a SQL expert in PostgreSQL.
Generate a PL/pgSQL query based on the schema.
Do not explain anything.
Only produce raw SQL – no markdown, no code fences.

Schema:
{schema}
"""),
    ("human", "{question}"),
])
sql_chain = sql_prompt | streaming_llm

# --- Full mode: intro message ---
intro_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful data analyst assistant. You MUST respond only in Persian (Farsi).
Write a short Persian introductory sentence (1-2 sentences max) that tells the user
you are about to show them a SQL query for their request.
Be natural and vary the phrasing. Do not produce any SQL or markdown.
"""),
    ("human", "{question}"),
])
intro_chain = intro_prompt | streaming_llm

# --- Full mode: sql explanation message ---
sql_message_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful data analyst assistant. You MUST respond only in Persian (Farsi).
Write a short Persian explanation (2-3 sentences) of what the provided SQL query does,
and end with a natural sentence indicating the result will follow.
Do not produce any SQL or markdown.
"""),
    ("human", "Question: {question}\n\nSQL: {sql}"),
])
sql_message_chain = sql_message_prompt | streaming_llm

# --- Full mode: result analysis ---
analyzer_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful data analyst.
You MUST always respond in Persian (Farsi) language only. Never respond in English.
Analyze the provided query result and write a concise, insightful Persian summary.
Always give answers with new lines and in separate paragraphs or organized lists.

Do not produce any Markdown.
Do not use any symbols like **, ###, ```, *, -, 1.
Produce only plain text.
Always give the answer in several separate paragraphs or as a list,
each item on a separate line, without any Markdown characters.
Separate lines by going to the next line.
"""),
    ("human", "Question: {question}\n\nSQL: {sql}\n\nResult: {result}"),
])
analyzer_chain = analyzer_prompt | streaming_llm

# ─────────────────────────────────────────
# Nodes
# ─────────────────────────────────────────

# ── Intent (no streaming needed) ──────────────────────────────────────────────
def intent_node(state: AgentState):
    result = intent_chain.invoke({"question": state["question"]})
    return {"mode": result.mode}

# ── Router ────────────────────────────────────────────────────────────────────
def router(state: AgentState):
    return state["mode"]

# ── Chat (streaming) ──────────────────────────────────────────────────────────
def chat_node(state: AgentState, writer: StreamWriter):
    full_message = ""
    for chunk in chat_chain.stream({"question": state["question"]}):
        token = chunk.content
        if token:
            full_message += token
            writer({"type": "token", "node": "chat", "value": token})
    return {"message": full_message}

# ── SQL only (streaming) ──────────────────────────────────────────────────────
def sql_node(state: AgentState, writer: StreamWriter):
    """Mode = sql  →  stream the generated query, nothing else."""
    schema_text = get_db_schema_text()
    full_sql = ""
    for chunk in sql_chain.stream({"question": state["question"], "schema": schema_text}):
        token = chunk.content
        if token:
            full_sql += token
            writer({"type": "token", "node": "sql", "value": token})
    return {"sql": full_sql}

# ── Result  (sql streaming → execute without streaming) ───────────────────────
def result_node(state: AgentState, writer: StreamWriter):
    """
    Mode = result
    1. Stream the SQL generation tokens so the user can watch it appear.
    2. Wait until the full SQL is ready, then execute and return the result at once.
    """
    schema_text = get_db_schema_text()

    # Step 1 – stream SQL tokens
    full_sql = ""
    for chunk in sql_chain.stream({"question": state["question"], "schema": schema_text}):
        token = chunk.content
        if token:
            full_sql += token
            writer({"type": "token", "node": "sql", "value": token})

    # Step 2 – execute (blocking, no streaming)
    query_result = run_sql_query(full_sql)
    writer({"type": "result", "node": "result", "value": query_result})

    return {"sql": full_sql, "result": query_result}

# ── Full  (multi-step streaming) ─────────────────────────────────────────────
def full_node(state: AgentState, writer: StreamWriter):
    """
    Mode = full
    Order:
      1. intro_message  (stream)
      2. sql            (stream)
      3. sql_message    (stream)
      4. execute SQL    (blocking – emit result at once)
      5. analysis       (stream)
    """
    schema_text = get_db_schema_text()
    question    = state["question"]

    # ── 1. Intro message ──────────────────────────────────────────────────────
    writer({"type": "section_start", "node": "full", "section": "intro"})
    intro_text = ""
    for chunk in intro_chain.stream({"question": question}):
        token = chunk.content
        if token:
            intro_text += token
            writer({"type": "token", "node": "full", "section": "intro", "value": token})
    writer({"type": "section_end", "node": "full", "section": "intro"})

    # ── 2. SQL generation ─────────────────────────────────────────────────────
    writer({"type": "section_start", "node": "full", "section": "sql"})
    full_sql = ""
    for chunk in sql_chain.stream({"question": question, "schema": schema_text}):
        token = chunk.content
        if token:
            full_sql += token
            writer({"type": "token", "node": "full", "section": "sql", "value": token})
    writer({"type": "section_end", "node": "full", "section": "sql"})

    # ── 3. SQL explanation ────────────────────────────────────────────────────
    writer({"type": "section_start", "node": "full", "section": "sql_message"})
    sql_message_text = ""
    for chunk in sql_message_chain.stream({"question": question, "sql": full_sql}):
        token = chunk.content
        if token:
            sql_message_text += token
            writer({"type": "token", "node": "full", "section": "sql_message", "value": token})
    writer({"type": "section_end", "node": "full", "section": "sql_message"})

    # ── 4. Execute SQL (blocking) ─────────────────────────────────────────────
    query_result = run_sql_query(full_sql)
    writer({"type": "result", "node": "full", "section": "result", "value": query_result})

    # ── 5. Analysis (stream) ──────────────────────────────────────────────────
    writer({"type": "section_start", "node": "full", "section": "analysis"})
    analysis_text = ""
    for chunk in analyzer_chain.stream({"question": question, "sql": full_sql, "result": query_result}):
        token = chunk.content
        if token:
            analysis_text += token
            writer({"type": "token", "node": "full", "section": "analysis", "value": token})
    writer({"type": "section_end", "node": "full", "section": "analysis"})

    return {
        "sql":           full_sql,
        "result":        query_result,
        "intro_message": intro_text,
        "sql_message":   sql_message_text,
        "message":       analysis_text,
    }

# ─────────────────────────────────────────
# Graph
# ─────────────────────────────────────────
def build_graph():
    builder = StateGraph(AgentState)

    builder.add_node("intent", intent_node)
    builder.add_node("chat",   chat_node)
    builder.add_node("sql",    sql_node)
    builder.add_node("result", result_node)
    builder.add_node("full",   full_node)

    builder.set_entry_point("intent")

    builder.add_conditional_edges(
        "intent",
        router,
        {
            "chat":   "chat",
            "sql":    "sql",
            "result": "result",
            "full":   "full",
        },
    )

    builder.add_edge("chat",   END)
    builder.add_edge("sql",    END)
    builder.add_edge("result", END)
    builder.add_edge("full",   END)

    return builder.compile()

graph = build_graph()

# ─────────────────────────────────────────
# Runner helpers
# ─────────────────────────────────────────
def stream_graph(question: str):
    """
    Unified runner: prints every event to stdout.
    Adapt this to your API / websocket layer as needed.
    """
    inputs = {"question": question}

    for event in graph.stream(inputs, stream_mode=["updates", "custom"]):
        mode, data = event

        if mode == "updates":
            # State snapshots (intent result, final node state, …)
            print("[UPDATE]", data)

        elif mode == "custom":
            # Streaming tokens / section markers / results
            print("[CUSTOM]", data)


# ─────────────────────────────────────────
# Quick demo
# ─────────────────────────────────────────
if __name__ == "__main__":
    QUESTIONS = {
        "chat":   "سلام! می‌تونی درباره قابلیت‌های خودت توضیح بدی؟",
        "sql":    "فقط کوئری SQL برای گرفتن ۱۰ سفارش آخر بنویس.",
        "result": "نتیجه کوئری ده تا سفارش آخر رو بده.",
        "full":   "کدوم محصولات بیشترین فروش رو داشتن؟",
    }

    # Change this to test different modes
    test_mode = "sql"
    print(f"\n{'='*60}")
    print(f"Testing mode: {test_mode}")
    print(f"Question: {QUESTIONS[test_mode]}")
    print(f"{'='*60}\n")

    stream_graph(QUESTIONS[test_mode])


Testing mode: sql
Question: فقط کوئری SQL برای گرفتن ۱۰ سفارش آخر بنویس.

[UPDATE] {'intent': {'mode': 'sql'}}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': 'SELECT'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': ' *'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': ' FROM'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': ' orders'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': ' ORDER'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': ' BY'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': ' created'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': '_at'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': ' DESC'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': ' LIMIT'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': ' '}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': '10'}
[CUSTOM] {'type': 'token', 'node': 'sql', 'value': ';'}
[UPDATE] {'sql': {'sql': 'SELECT * FROM orders ORDER BY created_at DESC LIMIT 10;'}}


In [15]:
import asyncio
from typing import Optional, Any
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END
from langgraph.types import StreamWriter

# ─────────────────────────────────────────
# Config
# ─────────────────────────────────────────
BASE_URL = "https://api.gapgpt.app/v1"
API_KEY  = "sk-s8KnoW59PPxeHBvyzENeVoEiH2QbiNm1PxJt20H586up5p8n"
MODEL    = "gpt-4o"

# ─────────────────────────────────────────
# LLMs
# ─────────────────────────────────────────
llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
    streaming=False,
)

streaming_llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
    streaming=True,
)

# ─────────────────────────────────────────
# Schemas
# ─────────────────────────────────────────
class IntentOutput(BaseModel):
    mode: str = Field(description="chat | sql | result | full")

# ─────────────────────────────────────────
# State
# ─────────────────────────────────────────
class AgentState(TypedDict):
    question:      str
    mode:          Optional[str]
    message:       Optional[str]   # chat node final message
    sql:           Optional[str]   # generated SQL
    result:        Optional[Any]   # query execution result
    intro_message: Optional[str]   # full mode intro
    sql_message:   Optional[str]   # full mode sql explanation

# ─────────────────────────────────────────
# Helpers  (stubs – replace with your real implementations)
# ─────────────────────────────────────────
def get_db_schema_text() -> str:
    """Return the database schema as plain text."""
    # TODO: replace with your real schema loader
    return """
    TABLE users (id SERIAL PRIMARY KEY, name TEXT, email TEXT, created_at TIMESTAMP);
    TABLE orders (id SERIAL PRIMARY KEY, user_id INT REFERENCES users(id), total NUMERIC, created_at TIMESTAMP);
    TABLE products (id SERIAL PRIMARY KEY, name TEXT, price NUMERIC, stock INT);
    TABLE order_items (id SERIAL PRIMARY KEY, order_id INT REFERENCES orders(id), product_id INT REFERENCES products(id), quantity INT);
    """

def run_sql_query(sql: str) -> Any:
    """Execute the SQL and return the result."""
    # TODO: replace with your real DB executor
    return [{"id": 1, "name": "Alice", "total": 250.0}]

# ─────────────────────────────────────────
# Prompts & Chains
# ─────────────────────────────────────────

# --- Intent ---
intent_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an intent classifier for a SQL agent.
Classify the user request into one of these modes:
chat   → greeting or unrelated to database
sql    → user explicitly asks for SQL query only
result → user wants only the raw result (no explanation)
full   → default for any data question (intro + sql + explanation + analysis)
"""),
    ("human", "{question}"),
])
intent_chain = intent_prompt | llm.with_structured_output(IntentOutput)

# --- Chat ---
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a SQL intelligent assistant.
You MUST always respond in Persian (Farsi) language only.
Only answer SQL and database-related questions.
"""),
    ("human", "{question}"),
])
chat_chain = chat_prompt | streaming_llm

# --- SQL generation (used in sql, result, full modes) ---
sql_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a SQL expert in PostgreSQL.
Generate a PL/pgSQL query based on the schema.
Do not explain anything.
Only produce raw SQL – no markdown, no code fences.

Schema:
{schema}
"""),
    ("human", "{question}"),
])
sql_chain = sql_prompt | streaming_llm

# --- Full mode: intro message ---
intro_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful data analyst assistant. You MUST respond only in Persian (Farsi).
Write a short Persian introductory sentence (1-2 sentences max) that tells the user
you are about to show them a SQL query for their request.
Be natural and vary the phrasing. Do not produce any SQL or markdown.
"""),
    ("human", "{question}"),
])
intro_chain = intro_prompt | streaming_llm

# --- Full mode: sql explanation message ---
sql_message_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful data analyst assistant. You MUST respond only in Persian (Farsi).
Write a short Persian explanation (2-3 sentences) of what the provided SQL query does,
and end with a natural sentence indicating the result will follow.
Do not produce any SQL or markdown.
"""),
    ("human", "Question: {question}\n\nSQL: {sql}"),
])
sql_message_chain = sql_message_prompt | streaming_llm

# --- Full mode: result analysis ---
analyzer_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful data analyst.
You MUST always respond in Persian (Farsi) language only. Never respond in English.
Analyze the provided query result and write a concise, insightful Persian summary.
Always give answers with new lines and in separate paragraphs or organized lists.

Do not produce any Markdown.
Do not use any symbols like **, ###, ```, *, -, 1.
Produce only plain text.
Always give the answer in several separate paragraphs or as a list,
each item on a separate line, without any Markdown characters.
Separate lines by going to the next line.
"""),
    ("human", "Question: {question}\n\nSQL: {sql}\n\nResult: {result}"),
])
analyzer_chain = analyzer_prompt | streaming_llm

# ─────────────────────────────────────────
# Nodes
# ─────────────────────────────────────────

# ── Intent (no streaming needed) ──────────────────────────────────────────────
def intent_node(state: AgentState):
    result = intent_chain.invoke({"question": state["question"]})
    return {"mode": result.mode}

# ── Router ────────────────────────────────────────────────────────────────────
def router(state: AgentState):
    return state["mode"]

# ── Chat (streaming) ──────────────────────────────────────────────────────────
def chat_node(state: AgentState, writer: StreamWriter):
    full_message = ""
    for chunk in chat_chain.stream({"question": state["question"]}):
        token = chunk.content
        if token:
            full_message += token
            writer({"type": "token", "node": "chat", "value": token})
    return {"message": full_message}

# ── SQL only (streaming) ──────────────────────────────────────────────────────
def sql_node(state: AgentState, writer: StreamWriter):
    """Mode = sql  →  stream the generated query, nothing else."""
    schema_text = get_db_schema_text()
    full_sql = ""
    for chunk in sql_chain.stream({"question": state["question"], "schema": schema_text}):
        token = chunk.content
        if token:
            full_sql += token
            writer({"type": "token", "node": "sql", "value": token})
    return {"sql": full_sql}

# ── Result  (sql streaming → execute without streaming) ───────────────────────
def result_node(state: AgentState, writer: StreamWriter):
    """
    Mode = result
    1. Stream the SQL generation tokens so the user can watch it appear.
    2. Wait until the full SQL is ready, then execute and return the result at once.
    """
    schema_text = get_db_schema_text()

    # Step 1 – stream SQL tokens
    full_sql = ""
    for chunk in sql_chain.stream({"question": state["question"], "schema": schema_text}):
        token = chunk.content
        if token:
            full_sql += token
            writer({"type": "token", "node": "sql", "value": token})

    # Step 2 – execute (blocking, no streaming)
    query_result = run_sql_query(full_sql)
    writer({"type": "result", "node": "result", "value": query_result})

    return {"sql": full_sql, "result": query_result}

# ── Full  (multi-step streaming) ─────────────────────────────────────────────
def full_node(state: AgentState, writer: StreamWriter):
    """
    Mode = full
    Order:
      1. intro_message  (stream)
      2. sql            (stream)
      3. sql_message    (stream)
      4. execute SQL    (blocking – emit result at once)
      5. analysis       (stream)
    """
    schema_text = get_db_schema_text()
    question    = state["question"]

    # ── 1. Intro message ──────────────────────────────────────────────────────
    writer({"type": "section_start", "node": "full", "section": "intro"})
    intro_text = ""
    for chunk in intro_chain.stream({"question": question}):
        token = chunk.content
        if token:
            intro_text += token
            writer({"type": "token", "node": "full", "section": "intro", "value": token})
    writer({"type": "section_end", "node": "full", "section": "intro"})

    # ── 2. SQL generation ─────────────────────────────────────────────────────
    writer({"type": "section_start", "node": "full", "section": "sql"})
    full_sql = ""
    for chunk in sql_chain.stream({"question": question, "schema": schema_text}):
        token = chunk.content
        if token:
            full_sql += token
            writer({"type": "token", "node": "full", "section": "sql", "value": token})
    writer({"type": "section_end", "node": "full", "section": "sql"})

    # ── 3. SQL explanation ────────────────────────────────────────────────────
    writer({"type": "section_start", "node": "full", "section": "sql_message"})
    sql_message_text = ""
    for chunk in sql_message_chain.stream({"question": question, "sql": full_sql}):
        token = chunk.content
        if token:
            sql_message_text += token
            writer({"type": "token", "node": "full", "section": "sql_message", "value": token})
    writer({"type": "section_end", "node": "full", "section": "sql_message"})

    # ── 4. Execute SQL (blocking) ─────────────────────────────────────────────
    query_result = run_sql_query(full_sql)
    writer({"type": "result", "node": "full", "section": "result", "value": query_result})

    # ── 5. Analysis (stream) ──────────────────────────────────────────────────
    writer({"type": "section_start", "node": "full", "section": "analysis"})
    analysis_text = ""
    for chunk in analyzer_chain.stream({"question": question, "sql": full_sql, "result": query_result}):
        token = chunk.content
        if token:
            analysis_text += token
            writer({"type": "token", "node": "full", "section": "analysis", "value": token})
    writer({"type": "section_end", "node": "full", "section": "analysis"})

    return {
        "sql":           full_sql,
        "result":        query_result,
        "intro_message": intro_text,
        "sql_message":   sql_message_text,
        "message":       analysis_text,
    }

# ─────────────────────────────────────────
# Graph
# ─────────────────────────────────────────
def build_graph():
    builder = StateGraph(AgentState)

    builder.add_node("intent", intent_node)
    builder.add_node("chat",   chat_node)
    builder.add_node("sql",    sql_node)
    builder.add_node("result", result_node)
    builder.add_node("full",   full_node)

    builder.set_entry_point("intent")

    builder.add_conditional_edges(
        "intent",
        router,
        {
            "chat":   "chat",
            "sql":    "sql",
            "result": "result",
            "full":   "full",
        },
    )

    builder.add_edge("chat",   END)
    builder.add_edge("sql",    END)
    builder.add_edge("result", END)
    builder.add_edge("full",   END)

    return builder.compile()

graph = build_graph()

# ─────────────────────────────────────────
# Runner
# ─────────────────────────────────────────
def run(question: str, streaming: bool = True):
    """
    Run the SQL agent graph.

    Parameters
    ----------
    question  : the user's question
    streaming : True  → yield / print token-by-token events (custom + updates)
                False → invoke the graph once and return the final state dict
    
    Returns
    -------
    streaming=True  → generator that yields (event_mode, event_data) tuples
    streaming=False → final AgentState dict
    """
    inputs = {"question": question}

    if streaming:
        # ── Streaming path ────────────────────────────────────────────────────
        # Nodes use StreamWriter → emits custom events token-by-token.
        # Caller can iterate the generator or just call run_and_print().
        def _event_generator():
            for event in graph.stream(inputs, stream_mode=["updates", "custom"]):
                yield event
        return _event_generator()

    else:
        # ── Non-streaming path ────────────────────────────────────────────────
        # graph.invoke() runs the graph to completion and returns the final state.
        # StreamWriter calls inside nodes are silently ignored (no-op) when the
        # graph is invoked without stream_mode, so no refactoring needed.
        final_state: AgentState = graph.invoke(inputs)
        return final_state


# ─────────────────────────────────────────
# Convenience print helper
# ─────────────────────────────────────────
def run_and_print(question: str, streaming: bool = True):
    """Run the graph and print results to stdout — useful for quick testing."""
    if streaming:
        for event_mode, event_data in run(question, streaming=True):
            if event_mode == "updates":
                print("[UPDATE]", event_data)
            elif event_mode == "custom":
                print("[CUSTOM]", event_data)
    else:
        final_state = run(question, streaming=False)
        print("[FINAL STATE]", final_state)


# ─────────────────────────────────────────
# Quick demo
# ─────────────────────────────────────────
if __name__ == "__main__":
    QUESTIONS = {
        "chat":   "سلام! می‌تونی درباره قابلیت‌های خودت توضیح بدی؟",
        "sql":    "فقط کوئری SQL برای گرفتن ۱۰ سفارش آخر بنویس.",
        "result": "نتیجه کوئری ده تا سفارش آخر رو بده.",
        "full":   "کدوم محصولات بیشترین فروش رو داشتن؟",
    }

    test_mode  = "chat"
    do_stream  = False          # ← True = streaming,  False = non-streaming

    print(f"\n{'='*60}")
    print(f"Mode: {test_mode}  |  streaming={do_stream}")
    print(f"Question: {QUESTIONS[test_mode]}")
    print(f"{'='*60}\n")

    run_and_print(QUESTIONS[test_mode], streaming=do_stream)


Mode: chat  |  streaming=False
Question: سلام! می‌تونی درباره قابلیت‌های خودت توضیح بدی؟

[FINAL STATE] {'question': 'سلام! می\u200cتونی درباره قابلیت\u200cهای خودت توضیح بدی؟', 'mode': 'chat', 'message': 'سلام! بله، من می\u200cتوانم به سوالات مربوط به SQL و پایگاه\u200cداده\u200cها پاسخ دهم. این شامل نوشتن و بهینه\u200cسازی کوئری\u200cهای SQL، توضیح مفاهیم پایگاه\u200cداده مانند نرمال\u200cسازی، ایندکس\u200cها، تراکنش\u200cها و موارد دیگر است. همچنین می\u200cتوانم در مورد طراحی پایگاه\u200cداده و بهترین شیوه\u200cها در مدیریت داده\u200cها راهنمایی کنم. اگر سوال خاصی دارید، خوشحال می\u200cشوم که کمک کنم!'}
